# Manipulasi Data dengan Pandas
**Mata Kuliah: Data Science**  
Firdaus — NIM 10124497

---

Implementasi pemrosesan dan manipulasi data tabular menggunakan pustaka Pandas sesuai spesifikasi penugasan:
- **Task 1:** Indexing & Selection (Pembersihan Anomali & Imputasi)
- **Task 2:** Operasi Aritmatika (Feature Engineering)
- **Task 3:** Filtering (Penyaringan Data)
- **Task 4:** Sorting (Pengurutan Data)
- **Task 5:** Grouping & Agregasi (Pelaporan Kinerja Gerai)

### Inisialisasi Dependensi dan Pemuatan Dataset
Memuat pustaka pandas dan numpy, serta membaca file transaksi ke dalam DataFrame.

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('data/paket_a_transaksi_pos_2024_2025.csv')
df.head()

,id_transaksi,tanggal,kode_gerai,kode_kasir,kode_produk,jumlah,harga_satuan,diskon_persen,status
0,TRX0241065,2025-06-02,G09,K011,RMT1402,4,13000,0,SELESAI
1,TRX0240510,2024-12-28,G09,K015,SGR1205,1,21000,0,SELESAI
2,TRX0244483,2024-09-04,G11,K029,RMT1407,3,"7500,00",0,SELESAI
3,TRX0242532,2025-07-30,G05,K001,MKN1103,2,6500,0,SELESAI
4,TRX0243509,26/03/2025,G05,K016,RMT1407,1,6000,0,SELESAI


### Task 1 — Indexing & Selection (Pembersihan Anomali & Imputasi)

**1. Pembersihan Format Harga**  
Pemeriksaan tipe data kolom `harga_satuan`, penggantian koma desimal menjadi titik, dan konversi ke tipe float.

In [2]:
print("Tipe data awal:", df['harga_satuan'].dtype)

df['harga_satuan'] = df['harga_satuan'].astype(str).str.replace(',', '.')
df['harga_satuan'] = pd.to_numeric(df['harga_satuan'], errors='coerce').astype(float)

print("Tipe data setelah konversi:", df['harga_satuan'].dtype)
df[['id_transaksi', 'harga_satuan']].head()

Tipe data awal: str
Tipe data setelah konversi: float64


,id_transaksi,harga_satuan
0,TRX0241065,13000.0
1,TRX0240510,21000.0
2,TRX0244483,7500.0
3,TRX0242532,6500.0
4,TRX0243509,6000.0


**2. Penanganan Kuantitas Negatif**  
Mengubah seluruh nilai kuantitas (`jumlah`) yang kurang dari 0 menjadi nilai absolut positif menggunakan Boolean Indexing bersama `.loc[]` dan `.abs()`.

In [3]:
df.loc[df['jumlah'] < 0, 'jumlah'] = df.loc[df['jumlah'] < 0, 'jumlah'].abs()

print("Nilai minimum kuantitas setelah perbaikan:", df['jumlah'].min())
df[['id_transaksi', 'jumlah']].head()

Nilai minimum kuantitas setelah perbaikan: 1


,id_transaksi,jumlah
0,TRX0241065,4
1,TRX0240510,1
2,TRX0244483,3
3,TRX0242532,2
4,TRX0243509,1


**3. Imputasi Nilai Kosong Kasir**  
Mengisi seluruh nilai kosong (NaN) pada kolom `kode_kasir` dengan label default `'K_UNKNOWN'` menggunakan `.loc[]`.

In [4]:
df.loc[df['kode_kasir'].isna(), 'kode_kasir'] = 'K_UNKNOWN'

print("Jumlah missing values kode_kasir:", df['kode_kasir'].isnull().sum())
df[['id_transaksi', 'kode_kasir']].head()

Jumlah missing values kode_kasir: 0


,id_transaksi,kode_kasir
0,TRX0241065,K011
1,TRX0240510,K015
2,TRX0244483,K029
3,TRX0242532,K001
4,TRX0243509,K016


**4. Seleksi Posisi Slicing**  
Menampilkan hanya kolom `id_transaksi`, `tanggal`, `kode_gerai`, dan `jumlah` untuk 5 baris pertama menggunakan metode `.iloc[]`.

In [5]:
kolom_target = ['id_transaksi', 'tanggal', 'kode_gerai', 'jumlah']
posisi_kolom = [df.columns.get_loc(col) for col in kolom_target]

df.iloc[:5, posisi_kolom]

,id_transaksi,tanggal,kode_gerai,jumlah
0,TRX0241065,2025-06-02,G09,4
1,TRX0240510,2024-12-28,G09,1
2,TRX0244483,2024-09-04,G11,3
3,TRX0242532,2025-07-30,G05,2
4,TRX0243509,26/03/2025,G05,1


### Task 2 — Operasi Aritmatika (Feature Engineering)

**1. Perhitungan Omzet Kotor**  
Membuat kolom `Total_Kotor` yang merupakan hasil perkalian vektor antara `harga_satuan` dan `jumlah`.

In [6]:
df['Total_Kotor'] = df['harga_satuan'] * df['jumlah']

df[['id_transaksi', 'harga_satuan', 'jumlah', 'Total_Kotor']].head()

,id_transaksi,harga_satuan,jumlah,Total_Kotor
0,TRX0241065,13000.0,4,52000.0
1,TRX0240510,21000.0,1,21000.0
2,TRX0244483,7500.0,3,22500.0
3,TRX0242532,6500.0,2,13000.0
4,TRX0243509,6000.0,1,6000.0


**2. Perhitungan Diskon**  
Membuat kolom `Nominal_Diskon` (`Total_Kotor` dikali persentase diskon desimal `diskon_persen / 100`).

In [7]:
df['Nominal_Diskon'] = df['Total_Kotor'] * (df['diskon_persen'] / 100)

df[['id_transaksi', 'Total_Kotor', 'diskon_persen', 'Nominal_Diskon']].head()

,id_transaksi,Total_Kotor,diskon_persen,Nominal_Diskon
0,TRX0241065,52000.0,0,0.0
1,TRX0240510,21000.0,0,0.0
2,TRX0244483,22500.0,0,0.0
3,TRX0242532,13000.0,0,0.0
4,TRX0243509,6000.0,0,0.0


**3. Perhitungan Omzet Bersih & Pajak**  
Membuat kolom `Total_Bersih` dari nilai omzet setelah diskon ditambah PPN 11%:  
$$\text{Total\_Bersih} = (\text{Total\_Kotor} - \text{Nominal\_Diskon}) \times 1.11$$

In [8]:
df['Total_Bersih'] = (df['Total_Kotor'] - df['Nominal_Diskon']) * 1.11

df[['id_transaksi', 'Total_Kotor', 'Nominal_Diskon', 'Total_Bersih']].head()

,id_transaksi,Total_Kotor,Nominal_Diskon,Total_Bersih
0,TRX0241065,52000.0,0.0,57720.0
1,TRX0240510,21000.0,0.0,23310.0
2,TRX0244483,22500.0,0.0,24975.0
3,TRX0242532,13000.0,0.0,14430.0
4,TRX0243509,6000.0,0.0,6660.0


### Task 3 — Filtering (Penyaringan Data)

**1. Filter Multi-Kondisi**  
Menyaring dan menampilkan transaksi dengan status `SELESAI`, gerai `G05` atau `G09`, dan `Total_Bersih` di atas Rp 100.000, serta mencetak jumlah baris data yang memenuhi kriteria.

In [9]:
filter_multi = df[
    (df['status'].astype(str).str.strip() == 'SELESAI') &
    (df['kode_gerai'].isin(['G05', 'G09'])) &
    (df['Total_Bersih'] > 100000)
]

print(f"Jumlah baris data yang memenuhi kriteria: {len(filter_multi):,} baris")
filter_multi[['id_transaksi', 'kode_gerai', 'status', 'Total_Bersih']].head()

Jumlah baris data yang memenuhi kriteria: 139 baris


,id_transaksi,kode_gerai,status,Total_Bersih
7,TRX0241251,G09,SELESAI,189810.0
24,TRX0244253,G09,SELESAI,135420.0
25,TRX0244999,G05,SELESAI,126540.0
90,TRX0240898,G05,SELESAI,154845.0
206,TRX0241868,G05,SELESAI,136530.0


**2. Filter dengan Query**  
Menggunakan metode `.query()` untuk menampilkan transaksi khusus gerai `G10` dengan kuantitas pembelian minimal 5 unit.

In [10]:
filter_query = df.query("kode_gerai == 'G10' and jumlah >= 5")

print(f"Jumlah transaksi gerai G10 (jumlah >= 5): {len(filter_query):,} baris")
filter_query[['id_transaksi', 'kode_gerai', 'kode_produk', 'jumlah', 'Total_Bersih']].head()

Jumlah transaksi gerai G10 (jumlah >= 5): 39 baris


,id_transaksi,kode_gerai,kode_produk,jumlah,Total_Bersih
83,TRX0243880,G10,MKN1101,12,606060.0
166,TRX0241592,G10,SGR1202,6,86580.0
351,TRX0242686,G10,BYI1502,12,346320.0
556,TRX0241396,G10,RMT1406,12,69597.0
879,TRX0243559,G10,SGR1206,6,313020.0


### Task 4 — Sorting (Pengurutan Data)

**1. Multi-Column Sorting**  
Mengurutkan DataFrame berdasarkan `kode_gerai` secara alfabetis naik (A–Z), kemudian berdasarkan `Total_Bersih` dari nilai tertinggi ke terendah (*Descending*).

In [11]:
df_sorted = df.sort_values(by=['kode_gerai', 'Total_Bersih'], ascending=[True, False])

df_sorted[['kode_gerai', 'id_transaksi', 'jumlah', 'Total_Bersih']].head(10)

,kode_gerai,id_transaksi,jumlah,Total_Bersih
244,G01,TRX0243257,1456,42020160.0
2956,G01,TRX0241367,12,672660.0
3588,G01,TRX0240608,12,626040.0
2937,G01,TRX0240631,12,559440.0
5,G01,TRX0240658,12,545454.0
3889,G01,TRX0241537,12,359640.0
2179,G01,TRX0244635,6,349650.0
4270,G01,TRX0240428,6,323010.0
695,G01,TRX0244706,6,313020.0
4404,G01,TRX0242179,6,313020.0


**2. Top-N Records**  
Mengambil 5 transaksi dengan nilai `Total_Bersih` paling tinggi di seluruh dataset menggunakan fungsi jalan pintas `.nlargest()`.

In [12]:
top_5_transaksi = df.nlargest(5, 'Total_Bersih')

top_5_transaksi[['id_transaksi', 'tanggal', 'kode_gerai', 'kode_produk', 'jumlah', 'Total_Bersih']]

,id_transaksi,tanggal,kode_gerai,kode_produk,jumlah,Total_Bersih
994,TRX0241002,2024-03-18,G11,MKN1101,2093,100363536.0
4985,TRX0244221,2024-12-09,G09,BYI1500,1301,66934498.5
467,TRX0243696,2024-02-27,G06,SGR1201,892,53466480.0
3393,TRX0241029,2025-05-12,G07,SGR1205,1983,45563391.0
244,TRX0243257,2024-11-24,G01,MKN1102,1456,42020160.0


### Task 5 — Grouping & Agregasi (Pelaporan Kinerja Gerai)

**1. Laporan Agregasi Per Gerai**  
Menyaring transaksi berstatus `SELESAI`, membuat tabel ringkasan `laporan_gerai` menggunakan `.groupby()` dan `.agg()`, mengurutkan dari `Total_Omzet_Bersih` tertinggi, serta membulatkan nilai desimal ke 2 digit.

In [13]:
df_selesai = df[df['status'].astype(str).str.strip() == 'SELESAI'].copy()

laporan_gerai = df_selesai.groupby('kode_gerai').agg(
    Jumlah_Transaksi=('id_transaksi', 'count'),
    Total_Unit_Terjual=('jumlah', 'sum'),
    Total_Omzet_Bersih=('Total_Bersih', 'sum'),
    Rata_Rata_Transaksi=('Total_Bersih', 'mean')
).sort_values(by='Total_Omzet_Bersih', ascending=False).round(2)

laporan_gerai.to_csv('output/laporan_gerai.csv')
laporan_gerai

,Jumlah_Transaksi,Total_Unit_Terjual,Total_Omzet_Bersih,Rata_Rata_Transaksi
kode_gerai,,,,
G11,398,3170,1.253536e+08,314958.82
G06,383,3103,9.113936e+07,237961.77
G09,379,3027,9.019606e+07,237984.32
G07,384,4284,8.036181e+07,209275.55
G01,450,2635,7.077959e+07,157287.97
G03,372,3274,5.980970e+07,160778.77
G10,401,1742,5.892248e+07,146938.85
G08,408,1700,3.550256e+07,87016.08
G12,406,1010,2.464483e+07,60701.55


**2. Transformasi Kelompok (Window Function)**  
Menggunakan `.groupby().transform()` untuk menambahkan kolom `Kontribusi_Gerai_Persen` yang menghitung persentase kontribusi `Total_Bersih` tiap transaksi terhadap total omzet gerainya:  
$$\text{Kontribusi\_Gerai\_Persen} = \left(\frac{\text{Total\_Bersih}}{\text{Total\_Omzet\_Gerai\_Tersebut}}\right) \times 100$$

In [14]:
total_omzet_gerai = df.groupby('kode_gerai')['Total_Bersih'].transform('sum')
df['Kontribusi_Gerai_Persen'] = ((df['Total_Bersih'] / total_omzet_gerai) * 100).round(2)

df[['kode_gerai', 'id_transaksi', 'Total_Bersih', 'Kontribusi_Gerai_Persen']].head(10)

,kode_gerai,id_transaksi,Total_Bersih,Kontribusi_Gerai_Persen
0,G09,TRX0241065,57720.0,0.06
1,G09,TRX0240510,23310.0,0.03
2,G11,TRX0244483,24975.0,0.02
3,G05,TRX0242532,14430.0,0.06
4,G05,TRX0243509,6660.0,0.03
5,G01,TRX0240658,545454.0,0.75
6,G10,TRX0244581,19980.0,0.03
7,G09,TRX0241251,189810.0,0.21
8,G07,TRX0244274,36630.0,0.04
9,G08,TRX0243163,14430.0,0.04
